<a href="https://colab.research.google.com/github/BlackBossX/Brain-Tumor-Classification/blob/main/notebooks/BrainTumorClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import pandas as pd
import numpy as np

#ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer


import tensorflow as tf
from tensorflow import keras


import matplotlib.pyplot as plt
import seaborn as sns

# Model Saving
import joblib


print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", __import__("sklearn").__version__)
print("TensorFlow:", tf.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)

Pandas: 2.2.2
NumPy: 2.0.2
Scikit-learn: 1.6.1
TensorFlow: 2.20.0
Matplotlib: 3.10.0
Seaborn: 0.13.2


In [35]:
df = pd.read_csv("/content/drive/MyDrive/BrainTumorClassification/brain_tumor_data.csv");

In [36]:
df.isna().sum()

,0
patient_id,0
age,0
gender,0
ethnicity,281
region,0
bmi,436
smoking_status,381
alcohol_consumption,3804
family_history,356
tumor_size_mm,0


In [37]:
df.head()

,patient_id,age,gender,ethnicity,region,bmi,smoking_status,alcohol_consumption,family_history,tumor_size_mm,...,ct_density,edema_grade,contrast_enhancement,ki67_index,bp_systolic,bp_diastolic,wbc_count,crp_level,genetic_marker_status,tumor_type
0,PT100001,60,Male,African,Rural,28.1,Never,Moderate,No,13.0,...,51.4,0.0,Mild,0.7,103,85.0,7.56,7.81,Positive,Meningioma
1,PT100002,50,Male,Asian,Urban,25.0,Never,Moderate,No,14.5,...,35.2,0.0,NaN,NaN,134,97.0,9.17,4.37,Positive,Pituitary
2,PT100003,74,Female,Other,Suburban,21.0,Never,NaN,No,10.7,...,43.0,1.0,Strong,3.4,127,62.0,3.67,10.16,Inconclusive,Meningioma
3,PT100004,34,Male,Hispanic,Suburban,29.2,Never,NaN,No,36.7,...,18.5,3.0,Mild,5.6,100,80.0,7.55,1.70,Positive,Glioma
4,PT100005,59,Male,Asian,Rural,28.2,Former,Moderate,No,16.0,...,24.0,1.0,Moderate,1.4,132,77.0,9.35,8.14,?,Pituitary


In [38]:
df.shape
#df.dtypes

(9000, 29)

In [39]:
#remove patient_id column cz not necessary for our predictions
df = df.drop("patient_id",axis=1)

In [40]:
df.head()

,age,gender,ethnicity,region,bmi,smoking_status,alcohol_consumption,family_history,tumor_size_mm,tumor_location,...,ct_density,edema_grade,contrast_enhancement,ki67_index,bp_systolic,bp_diastolic,wbc_count,crp_level,genetic_marker_status,tumor_type
0,60,Male,African,Rural,28.1,Never,Moderate,No,13.0,Cerebellum,...,51.4,0.0,Mild,0.7,103,85.0,7.56,7.81,Positive,Meningioma
1,50,Male,Asian,Urban,25.0,Never,Moderate,No,14.5,Sellar,...,35.2,0.0,NaN,NaN,134,97.0,9.17,4.37,Positive,Pituitary
2,74,Female,Other,Suburban,21.0,Never,NaN,No,10.7,Temporal,...,43.0,1.0,Strong,3.4,127,62.0,3.67,10.16,Inconclusive,Meningioma
3,34,Male,Hispanic,Suburban,29.2,Never,NaN,No,36.7,Occipital,...,18.5,3.0,Mild,5.6,100,80.0,7.55,1.70,Positive,Glioma
4,59,Male,Asian,Rural,28.2,Former,Moderate,No,16.0,Sellar,...,24.0,1.0,Moderate,1.4,132,77.0,9.35,8.14,?,Pituitary


In [41]:
df['alcohol_consumption'] = df['alcohol_consumption'].replace('Unknown', np.nan)
df['genetic_marker_status'] = df['genetic_marker_status'].replace('?', np.nan)


In [80]:
df["gender"] = df["gender"].str.strip().str.title()

In [81]:
df.alcohol_consumption
df.genetic_marker_status

,genetic_marker_status
0,Positive
1,Positive
2,Inconclusive
3,Positive
4,NaN
...,...
8995,Positive
8996,Positive
8997,Negative
8998,Positive


In [82]:
df.isna().sum()

,0
age,0
gender,0
ethnicity,281
region,0
bmi,436
smoking_status,381
alcohol_consumption,4185
family_history,356
tumor_size_mm,0
tumor_location,0


In [83]:
df.dtypes

,0
age,int64
gender,object
ethnicity,object
region,object
bmi,float64
smoking_status,object
alcohol_consumption,object
family_history,object
tumor_size_mm,float64
tumor_location,object


In [84]:
x = df.drop(columns='tumor_type')
y = df['tumor_type']


In [85]:
x.head()
y.head()

,tumor_type
0,Meningioma
1,Pituitary
2,Meningioma
3,Glioma
4,Pituitary


In [86]:
df['tumor_type'].value_counts()

,count
tumor_type,
Glioma,3802
Meningioma,3089
Pituitary,2109


In [87]:
X_train , X_temp , Y_train , Y_temp = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
    )

In [88]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=42,
    stratify=Y_temp
)

In [89]:
print(Y_train.value_counts(normalize=True) * 100)
print(y_val.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)


tumor_type
Glioma        42.253968
Meningioma    34.317460
Pituitary     23.428571
Name: proportion, dtype: float64
tumor_type
Glioma        42.222222
Meningioma    34.296296
Pituitary     23.481481
Name: proportion, dtype: float64
tumor_type
Glioma        42.222222
Meningioma    34.370370
Pituitary     23.407407
Name: proportion, dtype: float64


In [90]:
numeric_cols = X_train.select_dtypes(include='number').columns
categorical_cols = X_train.select_dtypes(exclude='number').columns

In [91]:
numeric_cols

Index(['age', 'bmi', 'tumor_size_mm', 'tumor_growth_rate', 'headache_severity',
       'mri_intensity', 'ct_density', 'edema_grade', 'ki67_index',
       'bp_systolic', 'bp_diastolic', 'wbc_count', 'crp_level'],
      dtype='object')

In [92]:
categorical_cols

Index(['gender', 'ethnicity', 'region', 'smoking_status',
       'alcohol_consumption', 'family_history', 'tumor_location', 'nausea',
       'vision_problems', 'seizures', 'memory_loss', 'balance_issues',
       'contrast_enhancement', 'genetic_marker_status'],
      dtype='object')

In [93]:
numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')

In [94]:
numeric_imputer.fit(X_train[numeric_cols])
categorical_imputer.fit(X_train[categorical_cols])

SimpleImputer(strategy='most_frequent')

In [95]:
X_train[numeric_cols] = numeric_imputer.transform(X_train[numeric_cols])
X_val[numeric_cols] = numeric_imputer.transform(X_val[numeric_cols])
X_test[numeric_cols] = numeric_imputer.transform(X_test[numeric_cols])

X_train[categorical_cols] = categorical_imputer.transform(X_train[categorical_cols])
X_val[categorical_cols] = categorical_imputer.transform(X_val[categorical_cols])
X_test[categorical_cols] = categorical_imputer.transform(X_test[categorical_cols])

In [96]:
for col in categorical_cols:
    print("=" * 40)
    print(col)
    print(X_test[col].unique())

gender
['Male' 'Female']
ethnicity
['Asian' 'Hispanic' 'Caucasian' 'African' 'Other']
region
['Urban' 'Suburban' 'Rural']
smoking_status
['Former' 'Never' 'Current']
alcohol_consumption
['Heavy' 'Moderate']
family_history
['No' 'Yes']
tumor_location
['Frontal' 'Parietal' 'Temporal' 'Sellar' 'Occipital' 'Convexity'
 'Cerebellum']
nausea
['Yes' 'No']
vision_problems
['Yes' 'No']
seizures
['Yes' 'No']
memory_loss
['No' 'Yes']
balance_issues
['No' 'Yes']
contrast_enhancement
['Moderate' 'Strong' 'Mild']
genetic_marker_status
['Negative' 'Positive' 'Inconclusive']


In [97]:
mapping = {"Male": 0, "Female": 1}

for dataset in [X_train, X_val, X_test]:
    dataset["gender"] = dataset["gender"].map(mapping)

In [100]:
mapping = {"Yes": 1, "No": 0}

for dataset in [X_train, X_val, X_test]:
    dataset["nausea"] = dataset["nausea"].map(mapping)
    dataset["vision_problems"] = dataset["vision_problems"].map(mapping)
    dataset["seizures"] = dataset["seizures"].map(mapping)
    dataset["memory_loss"] = dataset["memory_loss"].map(mapping)
    dataset["balance_issues"] = dataset["balance_issues"].map(mapping)
    dataset["family_history"] = dataset["family_history"].map(mapping)

In [101]:
for col in categorical_cols:
    print("=" * 40)
    print(col)
    print(X_test[col].unique())

gender
[0 1]
ethnicity
['Asian' 'Hispanic' 'Caucasian' 'African' 'Other']
region
['Urban' 'Suburban' 'Rural']
smoking_status
['Former' 'Never' 'Current']
alcohol_consumption
['Heavy' 'Moderate']
family_history
[0 1]
tumor_location
['Frontal' 'Parietal' 'Temporal' 'Sellar' 'Occipital' 'Convexity'
 'Cerebellum']
nausea
[1 0]
vision_problems
[1 0]
seizures
[1 0]
memory_loss
[0 1]
balance_issues
[0 1]
contrast_enhancement
['Moderate' 'Strong' 'Mild']
genetic_marker_status
['Negative' 'Positive' 'Inconclusive']
